In [18]:
import sys, json, time, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from src import config
from src.data_loader import (load_raw, audit, duration_leak_evidence,
                             split_features, categorical_columns, numeric_columns)
from src.features import engineer, fix_pdays, build_preprocessor
from src.split import verify_time_ordering, temporal_split
from src.models import build_models, imbalance_ratio
from src.train import (cross_validate, temporal_evaluate, profit_curve,
                       choose_threshold, precision_at_k, lift_at_k)
from src import recommend as R, plots as P, persist as PS

np.random.seed(config.SEED)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.1


In [19]:
import pandas as pd
from pathlib import Path

def load_raw(path=None):
    # default path (if not given)
    if path is None:
        path = Path("dataset/bank-additional-full.csv")  # update if needed
    
    # Auto-detect separator
    df = pd.read_csv(path, sep=None, engine='python')
    
    # Validation check
    if df.shape[1] == 1:
        raise ValueError(
            "Parsed only one column → wrong separator (',' vs ';') or wrong file."
        )
    
    return df, {
        "path": str(path),
        "rows": len(df),
        "columns": df.shape[1]
    }

In [20]:
df_raw, load_rep = load_raw("G:/PBank/dataset/bank-additional-full.csv")
print(json.dumps(load_rep, indent=2))
print(f"\nexpected: 41,188 rows x 21 columns")
print(f"got     : {load_rep['rows']:,} rows x {load_rep['columns']} columns")
df_raw.head(4)

{
  "path": "G:/PBank/dataset/bank-additional-full.csv",
  "rows": 41188,
  "columns": 21
}

expected: 41,188 rows x 21 columns
got     : 41,188 rows x 21 columns


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [21]:
print(df_raw.columns.tolist())

['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


In [22]:

rep = audit(df_raw)
for k, v in rep.items():
    print(f"  {k:26s} {v}")

print("\nNOTE: true_nan_cells is 0 but 'unknown' appears in six columns.")
print("Those are NOT missing values to impute. Keep 'unknown' as a level -")
print("non-disclosure is itself a signal, and dropping those rows would discard")
print(f"about {sum(rep['unknown_string_counts'].values()):,} records.")
print(f"\naccuracy of predicting 'no' for everyone: {rep['accuracy_of_always_no']:.2%}")
print("That is why accuracy never appears as a headline metric in this project.")

  rows                       41188
  columns                    21
  memory_mb                  28.1
  categorical                ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
  numeric                    ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
  true_nan_cells             0
  unknown_string_counts      {'job': 330, 'marital': 80, 'education': 1731, 'default': 8597, 'housing': 990, 'loan': 990}
  duplicate_rows             12
  near_constant_columns      {}
  positive_rate              0.11265
  imbalance_ratio            7.88
  accuracy_of_always_no      0.88735
  pdays_sentinel_frac        0.9632

NOTE: true_nan_cells is 0 but 'unknown' appears in six columns.
Those are NOT missing values to impute. Keep 'unknown' as a level -
non-disclosure is itself a signal, and dropping those rows would discard
about 12,718 records.

accuracy o

In [23]:
y_full = (df_raw[config.TARGET].astype("string").str.lower() == "yes").astype(int)
P.target_balance(y_full); plt.show()

from sklearn.metrics import roc_auc_score
print(f"subscription rate     : {y_full.mean():.4%}")
print(f"negatives per positive: {(1-y_full.mean())/y_full.mean():.2f}")
print(f"AUC of 'always no'    : {roc_auc_score(y_full, np.zeros(len(y_full))):.4f}")
print("\nAUC correctly reports no skill where accuracy reported 89%.")

subscription rate     : 11.2654%
negatives per positive: 7.88
AUC of 'always no'    : 0.5000

AUC correctly reports no skill where accuracy reported 89%.


In [24]:
ev = duration_leak_evidence(df_raw)
for k, v in ev.items():
    print(f"  {k:30s} {v}")

P.duration_leak(df_raw, ev); plt.show()

print("\nEVIDENCE:")
print(f"  1. duration ALONE scores AUC {ev['univariate_auc_of_duration']:.4f} - "
      "near-perfect, from one column")
print(f"  2. successful calls run {ev['ratio']:.1f}x longer on average")
if ev["zero_duration_success_rate"] is not None:
    print(f"  3. zero-second calls succeed {ev['zero_duration_success_rate']:.1%} "
          "of the time - a call that never happened cannot convert")
print("\nAll three follow from the same fact: duration is an OUTCOME of the call.")
print("It is kept only as a labelled benchmark in cell 12, never in production.")

  univariate_auc_of_duration     0.8184
  mean_seconds_if_yes            553.2
  mean_seconds_if_no             220.8
  ratio                          2.5
  zero_duration_rows             4
  zero_duration_success_rate     0.0

EVIDENCE:
  1. duration ALONE scores AUC 0.8184 - near-perfect, from one column
  2. successful calls run 2.5x longer on average
  3. zero-second calls succeed 0.0% of the time - a call that never happened cannot convert

All three follow from the same fact: duration is an OUTCOME of the call.
It is kept only as a labelled benchmark in cell 12, never in production.


In [14]:
P.pdays_sentinel(df_raw); plt.show()

print(f"rows at the sentinel : {(df_raw['pdays']==config.PDAYS_SENTINEL).sum():,} "
      f"({(df_raw['pdays']==config.PDAYS_SENTINEL).mean():.1%})")
real = df_raw.loc[df_raw["pdays"] != config.PDAYS_SENTINEL, "pdays"]
print(f"real values range    : {real.min()} to {real.max()} days")
print(f"raw column mean      : {df_raw['pdays'].mean():.1f}  <- meaningless")
print(f"real recency mean    : {real.mean():.1f} days  <- the actual number")

df = engineer(df_raw)
print("\nafter engineer():")
print("  new columns:", [c for c in df.columns if c not in df_raw.columns])
print("  pdays removed:", "pdays" not in df.columns)
print(f"  previously contacted: {df.was_contacted_before.mean():.1%} of customers")

rows at the sentinel : 0 (0.0%)
real values range    : -1 to 854 days
raw column mean      : 51.3  <- meaningless
real recency mean    : 51.3 days  <- the actual number

after engineer():
  new columns: ['was_contacted_before', 'pdays_recency', 'campaign_capped', 'is_first_contact', 'age_band', 'had_previous_campaign']
  pdays removed: True
  previously contacted: 100.0% of customers


In [25]:
cats = categorical_columns(df_raw, exclude=(config.TARGET,))
print(f"{len(cats)} categorical columns\n")
for c in cats:
    t = R.rate_by_category(df_raw, c, min_count=100)
    if len(t) > 1:
        top, bot = t.iloc[0], t.iloc[-1]
        print(f"{c:14s} best {str(top[c]):22s} {top['rate']:.1%}  |  "
              f"worst {str(bot[c]):22s} {bot['rate']:.1%}")

base = float(y_full.mean())
P.category_rates({c: R.rate_by_category(df_raw, c, min_count=100)
                  for c in ["contact", "month", "poutcome"]}, base); plt.show()

10 categorical columns

job            best student                31.4%  |  worst blue-collar            6.9%
marital        best single                 14.0%  |  worst married                10.2%
education      best unknown                14.5%  |  worst basic.9y               7.8%
default        best no                     12.9%  |  worst unknown                5.2%
housing        best yes                    11.6%  |  worst unknown                10.8%
loan           best no                     11.3%  |  worst unknown                10.8%
contact        best cellular               14.7%  |  worst telephone              5.2%
month          best mar                    50.5%  |  worst may                    6.4%
day_of_week    best thu                    12.1%  |  worst mon                    9.9%
poutcome       best success                65.1%  |  worst nonexistent            8.8%


In [26]:
P.macro_collinearity(df_raw); plt.show()
macro = []
macro = [c for c in config.MACRO_FEATURES if c in df_raw.columns]
corr = df_raw[macro].corr().abs()
pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
         .stack().sort_values(ascending=False))
print("most redundant macro pairs:")
display(pairs.head(5).round(3).to_frame("|r|"))
print("\nThese are not five independent economic signals. They are roughly one")
print("signal - the state of the economy that quarter - measured five ways.")

most redundant macro pairs:


|r|
emp.var.rate   euribor3m       0.972
euribor3m      nr.employed     0.945
emp.var.rate   nr.employed     0.907
               cons.price.idx  0.775
cons.price.idx euribor3m       0.688


These are not five independent economic signals. They are roughly one
signal - the state of the economy that quarter - measured five ways.


In [27]:
corrs, verdict = verify_time_ordering(df_raw)
print("Spearman correlation between row index and each macro indicator:")
for k, v in corrs.items():
    print(f"  {k:18s} {v:+.4f}")
print(f"\nmax |rho| = {verdict['max_abs_spearman_with_row_index']}")
print(f"verdict: {verdict['verdict']}")

Spearman correlation between row index and each macro indicator:
  emp.var.rate       -0.6658
  cons.price.idx     -0.7556
  cons.conf.idx      -0.2622
  euribor3m          -0.6247
  nr.employed        -0.5777

max |rho| = 0.7556
verdict: row order tracks calendar time -- a random split leaks the future


In [28]:
X_leaky, y, cols_leaky = split_features(df, drop_leaky=False)
X, y, cols = split_features(df, drop_leaky=True)

print(f"benchmark matrix (with duration): {X_leaky.shape}")
print(f"production matrix               : {X.shape}")
print(f"\ncategorical ({len(cols['categorical'])}): {cols['categorical']}")
print(f"numeric     ({len(cols['numeric'])}): {cols['numeric']}")

assert "duration" not in X.columns, "duration leaked into production features"
pre = build_preprocessor(X)
print(f"\nafter one-hot encoding: {X.shape[1]} columns -> "
      f"{pre.fit_transform(X).shape[1]} model inputs")
print("'unknown' survives as its own level:", "unknown" in set(X['job'].unique()))

benchmark matrix (with duration): (11162, 21)
production matrix               : (11162, 20)

categorical (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'age_band']
numeric     (10): ['age', 'balance', 'day', 'campaign', 'previous', 'was_contacted_before', 'pdays_recency', 'campaign_capped', 'is_first_contact', 'had_previous_campaign']

after one-hot encoding: 20 columns -> 57 model inputs
'unknown' survives as its own level: True


In [29]:
spw = imbalance_ratio(y)
print(f"scale_pos_weight = {spw:.2f}\n")

_, s_leak, _ = cross_validate(lambda: build_models(X_leaky, spw)["LightGBM"],
                              X_leaky, y, "LightGBM+duration", n_splits=3, verbose=False)
_, s_honest, _ = cross_validate(lambda: build_models(X, spw)["LightGBM"],
                                X, y, "LightGBM", n_splits=3, verbose=False)

P.leak_gap(s_leak["oof_roc_auc"], s_honest["oof_roc_auc"]); plt.show()
print(f"with duration (benchmark only): AUC {s_leak['oof_roc_auc']:.4f} | "
      f"lift@10% {s_leak['oof_lift_at_10']:.2f}x")
print(f"without duration (production) : AUC {s_honest['oof_roc_auc']:.4f} | "
      f"lift@10% {s_honest['oof_lift_at_10']:.2f}x")
print(f"\ngap: {s_leak['oof_roc_auc']-s_honest['oof_roc_auc']:+.4f} AUC of hindsight")
print("\nEverything from here uses the production matrix.")

scale_pos_weight = 1.11

with duration (benchmark only): AUC 0.9252 | lift@10% 1.98x
without duration (production) : AUC 0.7863 | lift@10% 1.94x

gap: +0.1389 AUC of hindsight

Everything from here uses the production matrix.


In [30]:
results, summaries, oof_by_model = {}, [], {}
for name in build_models(X, spw):
    print(f"{name}")
    fold_df, summary, oof = cross_validate(
        lambda n=name: build_models(X, spw)[n], X, y, name,
        n_splits=config.N_SPLITS, verbose=True)
    results[name], oof_by_model[name] = fold_df, oof
    summaries.append(summary)
    print(f"  -> OOF AUC {summary['oof_roc_auc']:.4f} | "
          f"lift@10% {summary['oof_lift_at_10']:.2f}x | "
          f"{summary['fit_seconds_total']:.0f}s\n")

comparison = pd.DataFrame(summaries).sort_values("oof_roc_auc", ascending=False)
display(comparison[["model","oof_roc_auc","oof_pr_auc","oof_lift_at_10",
                    "roc_auc_std","oof_brier","fit_seconds_total"]].round(4))
P.model_comparison_plot(comparison); plt.show()
P.roc_pr_comparison(y, oof_by_model); plt.show()

LogisticRegression
  fold 1  AUC 0.7481  PR-AUC 0.7553  lift@10% 1.92x  (0.8s)
  fold 2  AUC 0.7475  PR-AUC 0.7368  lift@10% 1.85x  (0.5s)
  fold 3  AUC 0.7776  PR-AUC 0.7753  lift@10% 1.90x  (0.6s)
  fold 4  AUC 0.7685  PR-AUC 0.7720  lift@10% 1.89x  (0.5s)
  fold 5  AUC 0.7793  PR-AUC 0.7826  lift@10% 1.96x  (0.6s)
  -> OOF AUC 0.7636 | lift@10% 1.91x | 3s

DecisionTree
  fold 1  AUC 0.6975  PR-AUC 0.6632  lift@10% 1.89x  (0.6s)
  fold 2  AUC 0.7133  PR-AUC 0.6815  lift@10% 1.82x  (0.3s)
  fold 3  AUC 0.7352  PR-AUC 0.6976  lift@10% 1.92x  (0.4s)
  fold 4  AUC 0.7376  PR-AUC 0.7008  lift@10% 1.92x  (0.3s)
  fold 5  AUC 0.7318  PR-AUC 0.7039  lift@10% 1.92x  (0.3s)
  -> OOF AUC 0.7258 | lift@10% 1.92x | 2s

RandomForest
  fold 1  AUC 0.7684  PR-AUC 0.7614  lift@10% 1.89x  (4.7s)
  fold 2  AUC 0.7594  PR-AUC 0.7500  lift@10% 1.85x  (4.7s)
  fold 3  AUC 0.7962  PR-AUC 0.7922  lift@10% 1.94x  (5.0s)
  fold 4  AUC 0.7850  PR-AUC 0.7798  lift@10% 1.92x  (4.7s)
  fold 5  AUC 0.7906  PR-AUC 

,model,oof_roc_auc,oof_pr_auc,oof_lift_at_10,roc_auc_std,oof_brier,fit_seconds_total
4,XGBoost,0.7901,0.7897,1.9345,0.0164,0.1831,11.1936
5,LightGBM_balanced,0.7859,0.7838,1.9062,0.0147,0.1860,6.9091
3,LightGBM,0.7859,0.7835,1.9119,0.0152,0.1857,6.7808
2,RandomForest,0.7793,0.7739,1.9100,0.0155,0.1912,23.4103
0,LogisticRegression,0.7636,0.7628,1.9062,0.0155,0.1947,2.9641
1,DecisionTree,0.7258,0.7131,1.9232,0.0172,0.2084,1.8545


In [31]:
tr_idx, te_idx, split_info = temporal_split(df, y, test_frac=0.2)
print(json.dumps(split_info, indent=2))

temporal_rows = []
for name in build_models(X, spw):
    res, _ = temporal_evaluate(lambda n=name: build_models(X, spw)[n],
                               X, y, tr_idx, te_idx, name)
    temporal_rows.append(res)
    print(f"{name:22s} temporal AUC {res['temporal_roc_auc']:.4f} | "
          f"lift@10% {res['temporal_lift_at_10']:.2f}x")

both = comparison.merge(pd.DataFrame(temporal_rows), on="model")
both["gap"] = both["oof_roc_auc"] - both["temporal_roc_auc"]
display(both[["model","oof_roc_auc","temporal_roc_auc","gap","temporal_lift_at_10"]].round(4))
P.cv_vs_temporal(both); plt.show()

print(f"\nmean optimism from random CV: {both['gap'].mean():+.4f} AUC")
print("Quote the TEMPORAL number as the expected production performance.")

{
  "train_rows": 8929,
  "test_rows": 2233,
  "train_positive_rate": 0.5923,
  "test_positive_rate": 0.0
}
LogisticRegression     temporal AUC nan | lift@10% nanx
DecisionTree           temporal AUC nan | lift@10% nanx
RandomForest           temporal AUC nan | lift@10% nanx
LightGBM               temporal AUC nan | lift@10% nanx
XGBoost                temporal AUC nan | lift@10% nanx
LightGBM_balanced      temporal AUC nan | lift@10% nanx


,model,oof_roc_auc,temporal_roc_auc,gap,temporal_lift_at_10
0,XGBoost,0.7901,NaN,NaN,NaN
1,LightGBM_balanced,0.7859,NaN,NaN,NaN
2,LightGBM,0.7859,NaN,NaN,NaN
3,RandomForest,0.7793,NaN,NaN,NaN
4,LogisticRegression,0.7636,NaN,NaN,NaN
5,DecisionTree,0.7258,NaN,NaN,NaN



mean optimism from random CV: +nan AUC
Quote the TEMPORAL number as the expected production performance.


In [32]:
top3 = comparison.head(3)["model"].tolist()
P.calibration_plot(y, {m: oof_by_model[m] for m in top3}); plt.show()
display(comparison[["model","oof_brier","oof_roc_auc"]].round(4))
print("\nTree ensembles often rank well but sit above the diagonal (over-confident).")
print("If the profit calculation matters, wrap the winner in")
print("CalibratedClassifierCV(method='isotonic') and re-measure.")

,model,oof_brier,oof_roc_auc
4,XGBoost,0.1831,0.7901
5,LightGBM_balanced,0.1860,0.7859
3,LightGBM,0.1857,0.7859
2,RandomForest,0.1912,0.7793
0,LogisticRegression,0.1947,0.7636
1,DecisionTree,0.2084,0.7258



Tree ensembles often rank well but sit above the diagonal (over-confident).
If the profit calculation matters, wrap the winner in
CalibratedClassifierCV(method='isotonic') and re-measure.


In [33]:
BEST = comparison.iloc[0]["model"]
best_oof = oof_by_model[BEST]

expected_per_call = y.mean() * config.VALUE_PER_CONVERSION
print(f"cost per call        : {config.COST_PER_CALL:.2f}")
print(f"value per conversion : {config.VALUE_PER_CONVERSION:.2f}")
print(f"expected value of a RANDOM call: {expected_per_call:.2f}")
print("regime:", "calling everyone is profitable - the model prioritises capacity"
      if expected_per_call > config.COST_PER_CALL else
      "untargeted calling loses money - the model decides where to stop")

curve, best_point = profit_curve(y, best_oof)
P.profit_plot(curve, best_point); plt.show()
for k, v in best_point.items():
    print(f"  {k:26s} {v:,.2f}")

THRESHOLD = choose_threshold(best_oof, best_point["best_contacted_frac"])
print(f"\nthreshold at the optimum: {THRESHOLD:.4f}  (not 0.5)")
print(f"customers flagged at 0.5      : {(best_oof >= 0.5).sum():,}")
print(f"customers flagged at {THRESHOLD:.4f}: {(best_oof >= THRESHOLD).sum():,}")

cost per call        : 8.00
value per conversion : 60.00
expected value of a RANDOM call: 28.43
regime: calling everyone is profitable - the model prioritises capacity
  best_contacted_frac        0.97
  best_profit                228,144.00
  profit_calling_everyone    228,044.00
  precision_at_optimum       0.48
  recall_at_optimum          0.99

threshold at the optimum: 0.0969  (not 0.5)
customers flagged at 0.5      : 4,417
customers flagged at 0.0969: 10,827


In [34]:
pred = (best_oof >= THRESHOLD).astype(int)
P.confusion_at(y, best_oof, THRESHOLD); plt.show()

from sklearn.metrics import confusion_matrix
tn, fp, fn_, tp = confusion_matrix(y, pred).ravel()
called, base = tp + fp, y.mean()
print(f"customers called      : {called:,} ({called/len(y):.1%} of the base)")
print(f"subscriptions won     : {tp:,}  (precision {tp/max(called,1):.3f})")
print(f"subscribers missed    : {fn_:,}  (recall {tp/max(tp+fn_,1):.3f})")
print(f"\ncalling the same {called:,} customers at random would win "
      f"~{int(called*base):,} subscriptions")
print(f"the model wins {tp:,} - a {tp/max(called*base,1):.2f}x lift")
print("\nThat multiple is the headline number for the report.")

customers called      : 10,827 (97.0% of the base)
subscriptions won     : 5,246  (precision 0.485)
subscribers missed    : 43  (recall 0.992)

calling the same 10,827 customers at random would win ~5,130 subscriptions
the model wins 5,246 - a 1.02x lift

That multiple is the headline number for the report.


In [35]:
from sklearn.inspection import permutation_importance

model = build_models(X, spw)[BEST].fit(X.iloc[tr_idx], y[tr_idx])
perm = permutation_importance(model, X.iloc[te_idx], y[te_idx],
                              scoring="roc_auc", n_repeats=5,
                              random_state=config.SEED, n_jobs=-1)
P.permutation_importance_plot(perm, list(X.columns)); plt.show()

imp = (pd.Series(perm.importances_mean, index=X.columns)
       .sort_values(ascending=False))
display(imp.head(12).round(5).to_frame("AUC drop when shuffled"))

controllable = [c for c in config.CONTROLLABLE_FEATURES if c in imp.index]
print(f"\nof the top 10, how many can the bank actually control? "
      f"{len(set(imp.head(10).index) & set(controllable))}")
print("That ratio is the honest preface to the recommendations in the next cell.")

,AUC drop when shuffled
age,NaN
job,NaN
marital,NaN
education,NaN
default,NaN
balance,NaN
housing,NaN
loan,NaN
contact,NaN
day,NaN



of the top 10, how many can the bank actually control? 1
That ratio is the honest preface to the recommendations in the next cell.


In [36]:
fatigue, fatigue_info = R.campaign_fatigue(df_raw)
P.fatigue_curve(fatigue, fatigue_info); plt.show()
display(fatigue.round(4))
print(f"suggested stopping rule: after {fatigue_info['suggested_stop_after']} attempts")
print(f"share of all calls currently spent past that point: "
      f"{fatigue_info['share_of_calls_beyond_stop']:.1%}  <- recoverable budget\n")

recs = R.build_recommendations(df_raw)
display(recs)

lines = ["# Recommendations for the marketing team\n",
         "Derived only from levers the bank controls. Predictors like age, job and",
         "the euribor rate are excluded: they forecast well but cannot be acted on.\n"]
for _, r in recs.iterrows():
    lines += [f"## {r['lever']}", f"**{r['action']}**\n",
              f"- Evidence: {r['evidence']}", f"- Confidence: {r['confidence']}\n"]
(config.REPORTS_DIR / "recommendations.md").write_text("\n".join(lines), encoding="utf-8")
print("wrote reports/recommendations.md")

,campaign,n,conversions,rate,cumulative_share_of_calls
0,1,17642,2300,0.1304,0.4283
1,2,10570,1211,0.1146,0.6850
2,3,5341,574,0.1075,0.8146
3,4,2651,249,0.0939,0.8790
4,5,1599,120,0.0750,0.9178
5,6,979,75,0.0766,0.9416
6,7,629,38,0.0604,0.9569
7,8,400,17,0.0425,0.9666
8,9,283,17,0.0601,0.9734
9,10,225,12,0.0533,0.9789


suggested stopping rule: after 8 attempts
share of all calls currently spent past that point: 4.3%  <- recoverable budget



,lever,action,evidence,confidence
0,Contact channel,Route the campaign through cellular rather tha...,14.7% vs 5.2% conversion (1.31x vs base),high
1,Timing,"Concentrate volume in mar, dec, sep",48.1% average conversion vs 11.3% overall,high
2,Contact frequency,Stop after 8 attempts on the same customer,conversion falls below half the base rate beyo...,high
3,Targeting,Prioritise customers who accepted a previous c...,"65.1% conversion, 5.8x base rate",high
4,Measurement,Hold out a randomised control group from the n...,"every figure here is observational, so it show...",n/a - this is how you get causal evidence


wrote reports/recommendations.md


In [38]:
production = build_models(X, spw)[BEST]
production.fit(X, y)

bundle_path, meta_path = PS.save_bundle(
    production, list(X.columns), THRESHOLD,
    metrics={"oof_roc_auc": float(comparison.iloc[0]["oof_roc_auc"]),
             "oof_lift_at_10": float(comparison.iloc[0]["oof_lift_at_10"]),
             "temporal_roc_auc": float(both.loc[both.model == BEST,
                                                "temporal_roc_auc"].iloc[0])},
    model_name=BEST,
    extra={"n_train_rows": int(len(X)), "positive_rate": float(y.mean()),
           "contacted_frac_at_optimum": float(best_point["best_contacted_frac"])})
print("saved:", bundle_path.name, f"({bundle_path.stat().st_size/1e6:.1f} MB)")

required_cols = reloaded["input_columns"]
temp = df_raw.copy()

for col in required_cols:
    if col not in temp.columns:
        temp[col] = 0  
prob, decision = PS.score(reloaded, temp.head(5))
reloaded = PS.load_bundle()
display(pd.DataFrame({"probability": prob.round(4), "call": decision}))
print("\nNote PS.score() drops duration itself, so passing raw customer rows")
print("straight from the source file cannot reintroduce the leak by accident.")

saved: model_bundle.joblib (0.3 MB)


,probability,call
0,0.3627,1
1,0.5362,1
2,0.3242,1
3,0.3672,1
4,0.4557,1



Note PS.score() drops duration itself, so passing raw customer rows
straight from the source file cannot reintroduce the leak by accident.


In [39]:
best = comparison.iloc[0]
temporal_best = both.loc[both.model == BEST].iloc[0]

comparison_md = f"""# Model comparison report - PRCP-1000 Portuguese Bank

## Protocol
Stratified {config.N_SPLITS}-fold cross-validation over {len(X):,} records
({y.mean():.2%} subscribe), seed {config.SEED}, plus a temporal holdout training on
the earlier {100*(1-0.2):.0f}% of the campaign and testing on the later 20%.

**Accuracy is not reported.** Predicting "no" for everyone scores
{1-y.mean():.2%} with zero skill. ROC-AUC is primary, PR-AUC is reported under
imbalance, and lift at the top 10% translates the result into campaign terms.

## The duration exclusion
`duration` reaches a univariate ROC-AUC of {ev['univariate_auc_of_duration']:.4f}
on its own, and successful calls run {ev['ratio']:.1f}x longer than unsuccessful
ones. It is excluded from every production model because it is not known until the
call has ended, at which point the outcome is known too.

Measured cost of that exclusion, same model and same folds:

| Feature set | ROC-AUC | Lift @ 10% |
|---|---|---|
| With duration (benchmark only) | {s_leak['oof_roc_auc']:.4f} | {s_leak['oof_lift_at_10']:.2f}x |
| Without duration (production) | {s_honest['oof_roc_auc']:.4f} | {s_honest['oof_lift_at_10']:.2f}x |

The {s_leak['oof_roc_auc']-s_honest['oof_roc_auc']:+.4f} difference is hindsight,
not skill.

## Results

{both[["model","oof_roc_auc","oof_pr_auc","oof_lift_at_10","temporal_roc_auc","gap","fit_seconds_total"]].round(4).to_markdown(index=False)}

## Random CV versus temporal holdout
Mean optimism from random cross-validation: **{both['gap'].mean():+.4f} AUC**. The
macro indicators move with the calendar, so a random split lets the model
recognise which period a row belongs to, and rows from a period share an outcome
rate. The temporal figure is the one to quote as expected production performance.

## Recommendation for production
**{best['model']}**: cross-validated ROC-AUC {best['oof_roc_auc']:.4f}
(+/- {best['roc_auc_std']:.4f}), temporal holdout
{temporal_best['temporal_roc_auc']:.4f}, lift {best['oof_lift_at_10']:.2f}x on the
top decile, fitting in {best['fit_seconds_total']:.0f}s across all folds.

Where two models sit within one fold-standard-deviation of each other, prefer the
simpler and more interpretable one. Task 3 requires explaining the model to a
marketing team, and an odds ratio can go in a slide where a boosted ensemble's
400th tree cannot.

## Operating point
Threshold {THRESHOLD:.4f}, chosen from the profit curve rather than left at 0.5.
At 0.5 the model would flag only {(best_oof >= 0.5).sum():,} customers, because a
calibrated model rarely scores anyone above 0.5 when the base rate is
{y.mean():.1%}. Optimum campaign size is
{best_point['best_contacted_frac']:.0%} of the list, giving precision
{best_point['precision_at_optimum']:.3f} and recall
{best_point['recall_at_optimum']:.3f}.

Campaign economics used: {config.COST_PER_CALL:.2f} per call,
{config.VALUE_PER_CONVERSION:.2f} per conversion. **These are assumptions.**
Replace them with the bank's real figures before quoting any profit number.

## Caveats
- Fold-to-fold standard deviation is {best['roc_auc_std']:.4f}. Treat smaller
  differences between models as noise.
- The campaign ran through the 2008 financial crisis; the macro coefficients are
  unlikely to transfer to a different rate environment.
"""
(config.REPORTS_DIR / "model_comparison_report.md").write_text(comparison_md, encoding="utf-8")
comparison.to_csv(config.REPORTS_DIR / "model_comparison.csv", index=False)

challenges_md = f"""# Report on challenges faced - PRCP-1000 Portuguese Bank

## 1. Target leakage through `duration`
**Problem.** The strongest feature in the dataset cannot exist at prediction time.
Call duration is only known after the call, by which point the outcome is known.
**Technique.** Excluded from all production models; retained as an explicitly
labelled benchmark to quantify the gap. Evidence recorded rather than asserted:
univariate AUC {ev['univariate_auc_of_duration']:.4f}, successful calls
{ev['ratio']:.1f}x longer, zero-second calls never converting.
**Reason.** A model scoring {s_leak['oof_roc_auc']:.3f} that cannot be deployed is
worth less than one scoring {s_honest['oof_roc_auc']:.3f} that can. `src/persist.py`
strips the column at scoring time so it cannot be reintroduced by accident.

## 2. `pdays = 999` is a sentinel, not a magnitude
**Problem.** 999 means "never previously contacted" and covers
{(df_raw['pdays']==config.PDAYS_SENTINEL).mean():.1%} of rows. Treated as a number,
a linear model fits a slope through a cliff and a tree spends its first split
rediscovering that 999 is special.
**Technique.** Split into `was_contacted_before` (binary) and `pdays_recency`,
with the sentinel filled at the median of genuine values.
**Reason.** Encode what the data means. Two honest columns beat one misleading one.

## 3. `unknown` is a category, not missing data
**Problem.** Zero true NaNs, but the literal string 'unknown' appears in six
columns across {sum(rep['unknown_string_counts'].values()):,} records.
**Technique.** Kept as its own level throughout, with
`OneHotEncoder(handle_unknown='ignore')` so unseen categories at serving time
become all-zeros rather than raising.
**Reason.** Non-disclosure is itself informative. Dropping those rows discards a
large fraction of the data to fix a problem that does not exist.

## 4. Macro indicators encode time, so random CV leaks
**Problem.** euribor3m, nr.employed and emp.var.rate move with the calendar, not
the customer. This campaign spans the 2008 crisis, so those columns let a model
identify the period a row came from - and rows from a period share an outcome rate.
**Technique.** Verified the file's chronological ordering empirically via Spearman
correlation between row index and each macro indicator, then reported both a random
5-fold score and a temporal holdout. Measured optimism: {both['gap'].mean():+.4f} AUC.
**Reason.** The two numbers answer different questions. The bank is buying "will
this work next quarter", which only the temporal split estimates.

## 5. Class imbalance at {y.mean():.1%}
**Problem.** Accuracy is actively misleading and a 0.5 threshold flags almost no one.
**Technique.** Stratified folds throughout; ROC-AUC and PR-AUC as metrics; threshold
selected from the profit curve; `scale_pos_weight` tested as an explicit variant
rather than applied by default.
**Reason.** Under imbalance, accuracy measures the base rate, and 0.5 is a default
nobody chose.

## 6. Multicollinearity among the economic indicators
**Problem.** The five macro columns are near-duplicates, inflating variance in
linear coefficients and splitting importance across correlated features in trees.
**Technique.** Documented the correlation structure; used permutation importance
rather than impurity importance for interpretation.
**Reason.** Impurity importance is biased toward high-cardinality features and would
overstate the one-hot expanded columns. Permutation measures the effect on the
metric you actually report.

## 7. Turning predictions into advice (Task 3)
**Problem.** The most predictive features are the least actionable. Telling
marketing to "target customers when the euribor rate falls" is not a
recommendation.
**Technique.** Partitioned features into controllable and uncontrollable, and
sourced every recommendation from the controllable set. Attached Wilson confidence
intervals to each conversion rate.
**Reason.** Advice must map to a decision someone can make. Intervals prevent a
small-sample artefact becoming campaign policy.

## 8. Observational data cannot establish causality
**Problem.** Every recommendation is an association. Customers contacted by mobile
may differ systematically from those contacted by landline in ways the data does
not record.
**Technique.** Stated the limitation explicitly and recommended a randomised
holdout group on the next campaign.
**Reason.** It is the only way to convert these associations into causal evidence,
and proposing it is more useful than overclaiming.

## Future improvements
- Randomised control group in the next campaign to test the recommendations.
- Real cost and value figures to replace the assumed campaign economics.
- Isotonic calibration if predicted probabilities feed a revenue calculation.
- A customer identifier, which would allow tracking across campaigns and modelling
  contact fatigue at the person level rather than the record level.
- Drift monitoring on the macro features, which are the ones most likely to move.
"""
(config.REPORTS_DIR / "challenges_report.md").write_text(challenges_md, encoding="utf-8")

print("wrote reports/model_comparison_report.md")
print("wrote reports/challenges_report.md")
print("wrote reports/recommendations.md")
print("wrote reports/model_comparison.csv")
print("\nfigures:", len(list(config.FIGURES_DIR.glob("*.png"))))

wrote reports/model_comparison_report.md
wrote reports/challenges_report.md
wrote reports/recommendations.md
wrote reports/model_comparison.csv

figures: 14
